# Mega stage evaluation, step 1: one big dump per photo set and condition
Every photo is drawn at a LARGE grid of headings x scales (default headings -15..15 step 2.5 = 13, scales 0.80..1.30 step 0.05 = 11, so 143 queries) through the pipeline's own coarse search, DINO re-rank (K = 50), LoFTR-MINIMA on M = 10 candidates per query and RANSAC. Every raw match is kept, so the replay notebook (`Mega_Replay_Figures.ipynb`) can score ANY subset of queries, K, M, bar, agreement rule and answer rule afterwards without the GPU.
One run = one condition (env `KIND`: clean, invert, blur, noise, lowres, gamma, yaw15, height20; `perturb.py` makes the altered photo, truth and sigma are never altered). Resumable: photos already in the dump folder are skipped.
Env: `GATE_DATA` (official or nudt), `NUDT_SPLIT`, `POOL_PHOTOS`, `PHOTOS_FILE` (csv with a photo column), `TAG`, `KIND`, `HEADINGS`, `SCALES` (comma lists), `N_LIMIT` (smoke test), `OUT_ROOT`.

In [1]:
import os, sys, time, functools
from pathlib import Path
import numpy as np, pandas as pd, torch
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
DATA = os.environ.get("GATE_DATA", "official")
POOL_PHOTOS = int(os.environ.get("POOL_PHOTOS", "500"))
PHOTOS_FILE = os.environ["PHOTOS_FILE"]
TAG = os.environ["TAG"]
KIND = os.environ.get("KIND", "clean")
N_LIMIT = int(os.environ.get("N_LIMIT", "0"))
HEADINGS = [float(v) for v in os.environ.get("HEADINGS", ",".join(str(-15 + 2.5 * i) for i in range(13))).split(",")]
SCALES = [float(v) for v in os.environ.get("SCALES", ",".join("%.2f" % (0.80 + 0.05 * i) for i in range(11))).split(",")]
OUT_ROOT = Path(os.environ.get("OUT_ROOT", str(ROOT / "tests/Test_Iteration_12/results")))
print(DATA, TAG, KIND, len(HEADINGS), "headings x", len(SCALES), "scales =", len(HEADINGS) * len(SCALES), "queries")

official official150 clean 13 headings x 11 scales = 143 queries


In [2]:
if DATA == "nudt":
    from train.datasets import photo_sets
    photo_sets.load_nudt_photos = functools.partial(photo_sets.load_nudt_photos, split=os.environ.get("NUDT_SPLIT", "test"))
from train.checks.rich_dump import RichDumpPipeline, pack_photo
from train.checks.perturb import perturbed_item
from image_processing.thermal import load_thermal_frame
pipeline = RichDumpPipeline(DATA, "edge_zncc", semantic_layer=True, semantic_decision=True, gamma=1.0, k_peaks=50, m_fine=10,
                            matcher="loftr_minima", min_inliers=8, ransac_pixels=None, matcher_confidence=0.22, satellite="new",
                            map_margin_m=None, loftr_coarse_threshold=0.21, dino_weight=None, headings=HEADINGS, scale_factors=SCALES)
t0 = time.time(); pipeline.setup(POOL_PHOTOS, 7); print("setup %.0f s" % (time.time() - t0))
test_set = pipeline.test_set
indices = sorted(int(i) for i in pd.read_csv(PHOTOS_FILE).photo)
if N_LIMIT:
    indices = indices[:N_LIMIT]
DUMP = OUT_ROOT / "rich_dump" / ("%s_%s" % (TAG, KIND)); DUMP.mkdir(parents=True, exist_ok=True)
done = {int(p.stem.split("_")[1]) for p in DUMP.glob("photo_*.npz")}
print(len(indices), "photos,", len(done & set(indices)), "done already ->", DUMP)

level 3: 0.85 m/pixel, picture 16378 x 16005, map coverage 100%
level 4: 1.20 m/pixel, picture 11581 x 11317, map coverage 100%
level 5: 1.70 m/pixel, picture 8189 x 8002, map coverage 100%
level 6: 2.40 m/pixel, picture 5790 x 5658, map coverage 100%
level 7: 3.39 m/pixel, picture 4094 x 4001, map coverage 100%


map store ready, took 99 s


/home/workstation/training/ideaforge/venv-geoloc/lib/python3.10/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


/home/workstation/training/ideaforge/venv-geoloc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


setup 128 s
150 photos, 47 done already -> /mnt/e/IdeaForge/work/mega/rich_dump/official150_clean


In [3]:
MAX_PHOTOS = int(os.environ.get("MAX_PHOTOS", "100000"))   # stop when this many photos of the set are in the dump folder
start = time.time(); failures = []; todo = [i for i in np.random.default_rng(5).permutation(indices).tolist() if i not in done]   # shuffled: any partial dump is a random sample
for count, index in enumerate(todo):
    if len(list(DUMP.glob("photo_*.npz"))) >= MAX_PHOTOS:
        break
    item = test_set.items[index] if KIND == "clean" else perturbed_item(test_set.items[index], KIND, str(OUT_ROOT / "perturbed_photos" / TAG), load_thermal_frame)
    try:
        found = None
        for attempt in range(3):
            try:
                found = pipeline.run_queries(item, test_set.priors[index]); break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); time.sleep(5)
        if found is None:
            raise MemoryError("GPU out of memory three times")
        arrays = pack_photo(item, test_set.truths[index], test_set.sigmas[index], found["level"], found["queries"], min_keep_confidence=0.15)
        arrays["height"] = float(test_set.items[index]["rel_height"]); arrays["yaw"] = float(test_set.items[index]["yaw"])
        np.savez_compressed(DUMP / ("photo_%d.npz" % index), **arrays)
    except Exception as error:
        failures.append((index, repr(error)[:200])); print("photo", index, "failed", repr(error)[:200])
    if count % 5 == 0:
        print("photo %d/%d | %.0f s | %.1f s per photo" % (count, len(todo), time.time() - start, (time.time() - start) / (count + 1)), flush=True)
pd.DataFrame(failures, columns=["photo", "error"]).to_csv(DUMP / "failures.csv", index=False)
print("done", len(todo), "photos in %.0f s," % (time.time() - start), len(failures), "failed")

photo 0/103 | 108 s | 107.8 s per photo


photo 5/103 | 611 s | 101.9 s per photo


photo 10/103 | 1092 s | 99.3 s per photo


photo 15/103 | 1609 s | 100.6 s per photo


photo 20/103 | 2075 s | 98.8 s per photo


photo 25/103 | 2530 s | 97.3 s per photo


photo 30/103 | 2964 s | 95.6 s per photo


done 103 photos in 3121 s, 0 failed
